# CONFIG

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import numpy as np
import os

from maikol_utils.print_utils import print_separator, print_warn, print_color

from src.config import Configuration    


CONFIG = Configuration(
    max_stages=15,
    objective_fpr=0.05
)

# seed all
np.random.seed(CONFIG.seed)

# GET ALL POSSIBLE HAAR

- We no dot create, for a single HAAR, both versions white-black / black-white, only one
- Since one is just the other $\times -1$, the weak classifiers will handle that by multiplying by $1$ or $-1$ as needed 

In [3]:
from src.data import generate_all_features

all_features = generate_all_features(
    win_w = CONFIG.crop_size, 
    win_h = CONFIG.crop_size
)
len(all_features)

86400

# LOAD DATA

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from maikol_utils.file_utils import load_json, list_dir_files

# =============== FACES DATASET =============== 
all_faces, n = list_dir_files(CONFIG.faces_path, recursive=True)
print(f" - Found {n} files in {CONFIG.faces_path}\n")


# =============== NO-FACES DATASET =============== 
to_keep_labels = load_json(CONFIG.dataset_classes_path)

# Download dataset without faces
fo.config.dataset_zoo_dir = CONFIG.no_faces_path
bg_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=to_keep_labels,
    max_samples=20000,
    # dataset_name="open-images-bg",  # ADDED: Forces a distinct dataset instance
    drop_existing_dataset=True      # ADDED: Clears old corrupted cache
)
bg_dataset = bg_dataset.filter_labels("ground_truth", F("label").is_in(to_keep_labels)) # REVERTED


 - Found 136965 files in ../data/ViolaJones/face_images

Loading output from ../data/dataset_classes.json...
  82% |████|-|    3.9Gb/4.8Gb [11.9s elapsed, 2.7s remaining, 323.6Mb/s]    

### PRECOMPUTE ALL FEATURES FOR FACES DATASET

In [ ]:
all_faces = np.random.choice(all_faces, size=100, replace=False)
print(f"Using {len(all_faces)} files in {CONFIG.faces_path}")

Using 100 files in ../data/ViolaJones/face_images


In [ ]:
from src.data import compute_features_dataset, precompute_feature_tensors


if os.path.exists(CONFIG.faces_np_path):
    print(f" - Loading precomputed face features from {CONFIG.faces_np_path}...")
    X_train_faces = np.load(CONFIG.faces_np_path)
    precomputed = precompute_feature_tensors(all_features)
    print(f" - Loaded face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")
else:
    X_train_faces, precomputed = compute_features_dataset(all_faces, all_features)
    np.save(CONFIG.faces_np_path, X_train_faces)
    print(f" - Computed face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")


 - Loading precomputed face features from ../data/ViolaJones/faces.npy...
 - Loaded face features for 5000 images.
 - X_train_faces dtype=float32, shape=(5000, 86400)


# VIOLA JONES STAGES

### Train AdaBoost
- Train with stumps, using at most x features per stage
- Then check at which point of the trained, we mee the True positive and False positive rate
- Remove unused stages 
- Return remaining tree and threshold for that stage

In [ ]:
from scripts import train_stage_early_stopping

### Generate hard mining samples
- Load random images without faces and get crops that PASS all the current stages


In [ ]:
from src.data import balance_non_face_samples

### Generate all cascade stages
2 stopping criteria 
- reaching FPR of $10 e -6$. This is computed not as in the paper (they estimated it from each stage $fpr_i$), but rather from the number of windows that had to be scanned to create the new 'X_train_bg'.
- Hand number (say 50)


In [ ]:
from scripts import generate_all_stages

stages, fpr_macro = generate_all_stages(
    CONFIG,
    X_train_faces=X_train_faces,
    bg_samples=[sample.filepath for sample in bg_dataset],
    all_features=all_features,
    precomputed=precomputed
)

________________________________________________________________________________________________________________________________
                                                      Training stage 1/15                                                       

________________________________________________________________
                Generating hard negative samples                

 - Generating 5000 hard negative samples (8 workers)...


Hard negatives:   0%|          | 0/5000 [00:00<?, ?crops/s]

# FINAL STAGES ClASSIFIER

### Save the trained cascade to XML

In [ ]:
from src.model import CascadeSerializer, build_haar_cascade_from_stages

haar_cascade = build_haar_cascade_from_stages(
    stages_output=stages,
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)

print_separator(f"FINAL CASCADE", sep_type="LONG")
print(f" - Stages: {len(haar_cascade.stages)}")
print(f" - Features used: {len(haar_cascade.features)}")
print(f" - Window size: {haar_cascade.width}x{haar_cascade.height}")
print(haar_cascade)


cascade_path = os.path.join(CONFIG.computed_haar_cascades, f"stages_vj-{fpr_macro:.4f}.xml")
CascadeSerializer.save(haar_cascade, cascade_path)

print(f" - File: {cascade_path}")

In [ ]:
loaded_cascade = CascadeSerializer.load(cascade_path)